# 🧠📊 PySpark Data Insights Agent
### Week 3 Take-Home Project — Google ADK Learning Series (Session 9 briefing)

**What this notebook does**

An **agent** that:
1. Accepts a **natural-language analytics question** (e.g., *"What are the top 5 sub-categories by total sales in the West region?"*).
2. Uses a **tool** to translate that question into a **parameterized PySpark aggregation** over a larger CSV / Parquet dataset (loaded from an **external link**).
3. Returns a **narrated summary + a chart**.
4. *(Optional)* Uses **Memory Bank–style persistent memory** so it can recall a user's previously asked questions across sessions.

**Design notes (per the session briefing)**
- Data is pulled from an **external raw link** (the recommended approach). A **synthetic fallback** dataset is generated automatically if the link is unreachable, so the notebook always runs end-to-end during grading. 🔁
- The core engine (Spark session → parameterized aggregation functions → plotting) works **without any API key**. The ADK `LlmAgent` layer is added on top and is **optional**.
- ⚠️ Colab sessions are ephemeral — **back this notebook up on a public GitHub repo** and submit the repo link.

**Submission**
- ✅ Colab notebook (link or `.ipynb` export)
- ✅ Sample Q&A transcript (bottom of this notebook, also exported separately)


---
## 1  Environment setup

`pyspark` and `matplotlib` are all that's required for the core agent.
`google-adk` is installed too but only used in the **optional** ADK section.

In [ ]:
# Core requirements (fast). ADK is optional and only used later.
!pip -q install pyspark==3.5.1 matplotlib pandas
# Optional: uncomment to enable the ADK LlmAgent layer (Section 6)
# !pip -q install google-adk
print("✅ Setup cell finished — restart NOT required.")

In [ ]:
import os, json, re, textwrap, datetime as dt
import matplotlib
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType, DateType)

# One Spark session for the whole notebook
spark = (SparkSession.builder
         .appName("PySparkDataInsightsAgent")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("✅ SparkSession ready:", spark.version)

---
## 2  Load the dataset (external link → Parquet, with a safe fallback)

Following the briefing, we **pull data from an external link**. Set `DATA_URL` to any
raw CSV you like (e.g., the `flights.csv` raw GitHub link shown in the session).

If the download fails (offline grading, expired link, etc.), the notebook **auto-generates
a realistic ~60,000-row `superstore_sales` dataset** with the same schema so every cell
still runs. We then write it to **Parquet** to show the CSV → Parquet path.

In [ ]:
# ---- Configure your external data source here -------------------------------
DATA_URL = "https://raw.githubusercontent.com/plotly/datasets/master/supermarket_Sales.csv"
CSV_PATH     = "sales_data.csv"
PARQUET_PATH = "sales_data.parquet"

# Canonical schema the agent understands
CANONICAL_COLS = ["order_date", "region", "segment", "category",
                  "sub_category", "sales", "quantity", "profit", "discount"]

def try_download(url, dest):
    """Attempt to pull the external CSV. Returns True on success."""
    try:
        import urllib.request
        urllib.request.urlretrieve(url, dest)
        # sanity check it is non-trivial
        if os.path.getsize(dest) > 1000:
            print(f"✅ Downloaded external dataset from:\n   {url}")
            return True
    except Exception as e:
        print(f"⚠️  External download failed ({type(e).__name__}): {e}")
    return False

def generate_fallback(dest, n_rows=60000, seed=7):
    """Deterministic synthetic 'superstore' sales data — guarantees the notebook runs."""
    import random, csv
    random.seed(seed)
    regions   = ["West", "East", "Central", "South"]
    segments  = ["Consumer", "Corporate", "Home Office"]
    catmap = {
        "Furniture":       ["Chairs", "Tables", "Bookcases", "Furnishings"],
        "Office Supplies": ["Binders", "Storage", "Paper", "Art", "Appliances"],
        "Technology":      ["Phones", "Machines", "Accessories", "Copiers"],
    }
    start = dt.date(2023, 1, 1)
    with open(dest, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(CANONICAL_COLS)
        for _ in range(n_rows):
            cat = random.choice(list(catmap))
            sub = random.choice(catmap[cat])
            day = start + dt.timedelta(days=random.randint(0, 730))
            base = {"Furniture": 350, "Office Supplies": 60, "Technology": 500}[cat]
            sales = round(max(5.0, random.gauss(base, base * 0.6)), 2)
            qty   = random.randint(1, 12)
            disc  = random.choice([0, 0, 0.1, 0.15, 0.2, 0.3])
            profit = round(sales * random.uniform(-0.15, 0.35) - sales * disc * 0.5, 2)
            w.writerow([day.isoformat(), random.choice(regions),
                        random.choice(segments), cat, sub,
                        sales, qty, profit, disc])
    print(f"✅ Generated fallback dataset ({n_rows:,} rows) at {dest}")

# 1) get a CSV (external first, fallback second)
if not try_download(DATA_URL, CSV_PATH):
    generate_fallback(CSV_PATH)


In [ ]:
# 2) Read CSV with Spark, normalise columns to the canonical schema, write Parquet
raw = (spark.read.option("header", True).option("inferSchema", True).csv(CSV_PATH))

# Map common alternate column names -> canonical names (robust to external files)
rename = {}
for c in raw.columns:
    key = c.strip().lower().replace(" ", "_").replace("-", "_")
    alias = {
        "order_date": "order_date", "date": "order_date",
        "region": "region", "segment": "segment",
        "category": "category", "product_category": "category",
        "sub_category": "sub_category", "subcategory": "sub_category",
        "sales": "sales", "amount": "sales", "revenue": "sales",
        "quantity": "quantity", "qty": "quantity",
        "profit": "profit", "discount": "discount",
    }.get(key)
    if alias:
        rename[c] = alias

df = raw
for old, new in rename.items():
    df = df.withColumnRenamed(old, new)

# Keep only canonical columns that exist; cast types sensibly
keep = [c for c in CANONICAL_COLS if c in df.columns]
df = df.select(*keep)
if "order_date" in df.columns:
    df = df.withColumn("order_date", F.to_date("order_date"))
for numc in ["sales", "profit", "discount"]:
    if numc in df.columns:
        df = df.withColumn(numc, F.col(numc).cast("double"))
if "quantity" in df.columns:
    df = df.withColumn("quantity", F.col("quantity").cast("int"))

df = df.cache()
row_count = df.count()

# Write to Parquet (demonstrates CSV -> Parquet) and reload from Parquet
df.write.mode("overwrite").parquet(PARQUET_PATH)
df = spark.read.parquet(PARQUET_PATH).cache()

print(f"✅ Loaded {row_count:,} rows | columns: {df.columns}")
df.printSchema()
df.show(5, truncate=False)

---
## 3  The tool: natural language → **parameterized** PySpark aggregation

The heart of the project. Instead of letting the model invent numbers (a failure mode
seen in the live session), the model's only job is to fill a **strict parameter object**;
**PySpark computes the real answer**.

**Parameter schema**

| field | meaning | example |
|---|---|---|
| `metric` | aggregation function | `sum`, `avg`, `count`, `min`, `max` |
| `measure` | numeric column to aggregate | `sales`, `profit`, `quantity` |
| `group_by` | dimension to group on | `region`, `category`, `sub_category`, `segment`, `month` |
| `filters` | optional equality filters | `{"region": "West"}` |
| `sort` | `asc` / `desc` | `desc` |
| `limit` | top-N rows | `5` |

A dependency-free **rule-based parser** maps a question to these params, so the tool
works with **no API key**. The optional ADK section can swap in an LLM to fill the same schema.

In [ ]:
# ---- Column registry: what the agent is allowed to touch --------------------
MEASURES   = [c for c in ["sales", "profit", "quantity", "discount"] if c in df.columns]
DIMENSIONS = [c for c in ["region", "segment", "category", "sub_category"] if c in df.columns]
if "order_date" in df.columns:
    DIMENSIONS = DIMENSIONS + ["month"]          # virtual time dimension
METRICS = ["sum", "avg", "count", "min", "max"]

# Known filter values (for simple filter detection), lowercased -> (col, value)
_filter_index = {}
for dim in [d for d in DIMENSIONS if d != "month"]:
    for row in df.select(dim).distinct().collect():
        v = row[dim]
        if v is not None:
            _filter_index[str(v).lower()] = (dim, v)

print("Measures  :", MEASURES)
print("Dimensions:", DIMENSIONS)
print("Metrics   :", METRICS)

In [ ]:
def _dim_variants(d):
    """Singular/plural/hyphen/space/underscore variants for a dimension name."""
    base = {d, d.replace("_", " "), d.replace("_", "")}
    plurals = set()
    for b in base:
        if b.endswith("y"):
            plurals.add(b[:-1] + "ies")     # category -> categories
        else:
            plurals.add(b + "s")            # region -> regions
    return base | plurals

# Precompute once
_DIM_VARIANTS = {d: _dim_variants(d) for d in DIMENSIONS if d != "month"}

def parse_question(question: str) -> dict:
    """Rule-based NL -> parameter object. Deterministic, no API key needed.

    Note: 'highest/lowest/top/bottom' describe SORT DIRECTION + limit, not the
    aggregation. The aggregation is sum by default, avg for 'average', count for 'how many'.
    """
    q = question.lower()
    qn = q.replace("-", " ")   # normalise 'sub-categories' -> 'sub categories'

    # ---- metric (aggregation) ----
    if any(w in q for w in ["average", "avg", "mean"]):
        metric = "avg"
    elif any(w in q for w in ["how many", "count", "number of", "how much"]) and "total" not in q:
        metric = "count"
    else:
        metric = "sum"                     # 'total', 'highest total', etc. all -> sum

    # ---- measure ----
    measure = "sales" if "sales" in MEASURES else (MEASURES[0] if MEASURES else "sales")
    if any(w in q for w in ["revenue", "sold", "sales"]) and "sales" in MEASURES:
        measure = "sales"
    if "profit" in q and "profit" in MEASURES:
        measure = "profit"
    if ("quantity" in q or "units" in q) and "quantity" in MEASURES:
        measure = "quantity"

    # ---- group_by dimension (handles plurals / hyphens) ----
    group_by = None
    if any(w in qn for w in ["month", "trend", "over time", "monthly"]) and "month" in DIMENSIONS:
        group_by = "month"
    else:
        # prefer the longest matching dimension name (sub_category before category)
        for d in sorted(_DIM_VARIANTS, key=lambda x: -len(x)):
            if any(re.search(r"\b" + re.escape(v) + r"\b", qn) for v in _DIM_VARIANTS[d]):
                group_by = d
                break

    # ---- filters (detect any known dimension value mentioned) ----
    filters = {}
    for val_lower, (col, val) in _filter_index.items():
        if re.search(r"\b" + re.escape(val_lower) + r"\b", qn):
            if col != group_by:            # don't filter the same field we group by
                filters[col] = val

    # ---- sort direction ----
    sort = "desc"
    if any(w in q for w in ["lowest", "smallest", "bottom", "least", "ascending", "fewest"]):
        sort = "asc"

    # ---- limit (top-N / bottom-N / 'which ... has the highest/lowest') ----
    limit = None
    m = re.search(r"(?:top|bottom|first|last)\s+(\d+)", q)
    if m:
        limit = int(m.group(1))
    elif any(w in q for w in ["top", "bottom"]):
        limit = 5
    elif q.strip().startswith(("which", "what")) and any(
            w in q for w in ["highest", "lowest", "most", "least", "best", "worst"]) and group_by:
        limit = 1                          # 'which category has the lowest total profit' -> 1 row

    return {"metric": metric, "measure": measure, "group_by": group_by,
            "filters": filters, "sort": sort, "limit": limit,
            "question": question}

In [ ]:
def run_aggregation(params: dict):
    """Execute a PySpark aggregation from a validated parameter object.
    Returns (pandas_df, resolved_params)."""
    metric   = params.get("metric", "sum")
    measure  = params.get("measure", "sales")
    group_by = params.get("group_by")
    filters  = params.get("filters", {}) or {}
    sort     = params.get("sort", "desc")
    limit    = params.get("limit")

    # ---- validate against the registry (guards against hallucinated columns) ----
    if metric not in METRICS:
        raise ValueError(f"Unsupported metric '{metric}'")
    if metric != "count" and measure not in MEASURES:
        raise ValueError(f"Unknown measure '{measure}'")
    if group_by is not None and group_by not in DIMENSIONS:
        raise ValueError(f"Unknown dimension '{group_by}'")

    data = df
    for col, val in filters.items():
        if col in data.columns:
            data = data.filter(F.col(col) == val)

    # virtual 'month' dimension from order_date
    if group_by == "month":
        data = data.withColumn("month", F.date_format("order_date", "yyyy-MM"))

    agg_col = F.count(F.lit(1)) if metric == "count" else getattr(F, metric)(F.col(measure))
    out_name = "count" if metric == "count" else f"{metric}_{measure}"
    agg_col = agg_col.alias(out_name)

    if group_by:
        res = data.groupBy(group_by).agg(agg_col)
        res = res.orderBy(F.col(out_name).asc() if sort == "asc" else F.col(out_name).desc())
        if limit:
            res = res.limit(limit)
    else:
        res = data.agg(agg_col)

    pdf = res.toPandas()
    if out_name in pdf.columns:
        pdf[out_name] = pdf[out_name].round(2)
    resolved = dict(params); resolved["out_name"] = out_name
    return pdf, resolved

---
## 4  Narration + chart

`narrate()` turns the numeric result into an English summary; `make_chart()` renders a bar
(categorical) or line (monthly trend) chart. Together with the tool above these are the
*"aggregation + plotting functions"* the briefing asks for.

In [ ]:
def narrate(pdf, resolved) -> str:
    """Human-readable summary of the aggregation result."""
    metric, measure = resolved["metric"], resolved["measure"]
    group_by, out = resolved.get("group_by"), resolved["out_name"]
    filt = resolved.get("filters") or {}
    label = "count of records" if metric == "count" else f"{metric} of {measure}"
    where = ("" if not filt else
             " (filtered to " + ", ".join(f"{k}={v}" for k, v in filt.items()) + ")")

    if not group_by:
        val = pdf.iloc[0][out]
        return f"The overall {label}{where} is **{val:,.2f}**."

    lines = [f"**{label.title()} by {group_by}{where}:**"]
    top = pdf.iloc[0]
    for _, r in pdf.iterrows():
        v = r[out]
        v = f"{v:,.2f}" if isinstance(v, float) else f"{v:,}"
        lines.append(f"  • {r[group_by]}: {v}")
    tv = top[out]; tv = f"{tv:,.2f}" if isinstance(tv, float) else f"{tv:,}"
    lines.append(f"\n👉 Highest is **{top[group_by]}** at **{tv}**.")
    return "\n".join(lines)


def make_chart(pdf, resolved):
    """Bar chart for categories, line chart for monthly trend."""
    group_by, out = resolved.get("group_by"), resolved["out_name"]
    if not group_by:
        print("(single value — no chart needed)"); return
    x = pdf[group_by].astype(str).tolist()
    y = pdf[out].tolist()
    plt.figure(figsize=(8, 4.5))
    if group_by == "month":
        order = sorted(range(len(x)), key=lambda i: x[i])
        x = [x[i] for i in order]; y = [y[i] for i in order]
        plt.plot(x, y, marker="o")
        plt.xticks(rotation=45, ha="right")
    else:
        plt.bar(x, y)
        plt.xticks(rotation=20, ha="right")
    plt.title(f"{out.replace('_',' ').title()} by {group_by}")
    plt.ylabel(out.replace("_", " ").title())
    plt.tight_layout(); plt.show()

---
## 5  Optional persistent memory (Memory Bank–style)

The briefing lists Memory Bank as **optional**. Here is a lightweight, dependency-free
implementation: questions are appended to a JSON file on disk so the agent can **recall a
user's earlier questions across sessions**. In a full ADK setup this would be backed by
`VertexAiMemoryBankService`; the interface (`remember` / `recall`) is kept identical so it
swaps in cleanly.

In [ ]:
MEMORY_FILE = "user_memory.json"

def _load_mem():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE) as f:
            return json.load(f)
    return {}

def remember(user_id: str, question: str, headline: str):
    mem = _load_mem()
    mem.setdefault(user_id, []).append({
        "ts": dt.datetime.now().isoformat(timespec="seconds"),
        "question": question, "headline": headline})
    with open(MEMORY_FILE, "w") as f:
        json.dump(mem, f, indent=2)

def recall(user_id: str, n: int = 3):
    return _load_mem().get(user_id, [])[-n:]

print("✅ Memory helpers ready (persist to user_memory.json).")

---
## 6  Put it together: the **agent**

`ask()` is the agent's entry point. It (1) recalls prior questions, (2) parses the NL
question into parameters, (3) runs the **real PySpark aggregation**, (4) narrates + charts,
and (5) remembers the question. This runs with **no API key**.

In [ ]:
def ask(question: str, user_id: str = "niraj", show_chart: bool = True, verbose: bool = True):
    print("🗣️  Q:", question)

    # 1) cross-session recall
    prior = recall(user_id)
    if prior and verbose:
        print("🧠 Memory — you previously asked:")
        for p in prior:
            print(f"     - ({p['ts']}) {p['question']}")

    # 2) NL -> params  (the 'tool')
    params = parse_question(question)
    if verbose:
        print("🔧 Tool params:", json.dumps({k: params[k] for k in
              ['metric','measure','group_by','filters','sort','limit']}, default=str))

    # 3) real PySpark aggregation
    pdf, resolved = run_aggregation(params)

    # 4) narrate + chart
    summary = narrate(pdf, resolved)
    print("\n📊 A:\n" + summary)
    if show_chart:
        make_chart(pdf, resolved)

    # 5) remember
    headline = summary.split("\n")[0]
    remember(user_id, question, headline)
    return summary

---
## 7  *(Optional)* Wrap it as a Google ADK `LlmAgent`

Everything above already satisfies the project. This **optional** cell exposes the same
aggregation as an **ADK `FunctionTool`** and lets an `LlmAgent` (Gemini) fill the parameter
schema instead of the rule-based parser. It only runs if `google-adk` is installed and an
API key is set — otherwise the notebook keeps using the rule-based `ask()` above.

In [ ]:
# Optional ADK layer — guarded so the notebook never breaks without keys.
ADK_READY = False
try:
    # os.environ["GOOGLE_API_KEY"] = "YOUR_AI_STUDIO_KEY"   # <- set this to enable
    if os.environ.get("GOOGLE_API_KEY"):
        from google.adk.agents import LlmAgent
        from google.adk.tools import FunctionTool

        def pyspark_insight(metric: str, measure: str = "sales", group_by: str = "",
                            region: str = "", top_n: int = 0) -> dict:
            """Run a PySpark aggregation over the sales dataset and return rows + a summary.

            Args:
                metric: one of sum, avg, count, min, max.
                measure: numeric column (sales, profit, quantity).
                group_by: dimension to group by (region, category, sub_category, segment, month) or empty.
                region: optional region filter (West/East/Central/South) or empty.
                top_n: limit to top-N rows (0 = no limit).
            """
            params = {"metric": metric, "measure": measure,
                      "group_by": group_by or None,
                      "filters": ({"region": region} if region else {}),
                      "sort": "desc", "limit": (top_n or None), "question": ""}
            pdf, resolved = run_aggregation(params)
            return {"summary": narrate(pdf, resolved),
                    "rows": pdf.to_dict(orient="records")}

        insight_tool = FunctionTool(func=pyspark_insight)
        data_agent = LlmAgent(
            name="pyspark_data_insights_agent",
            model="gemini-2.0-flash",
            instruction=("You answer analytics questions ONLY by calling the pyspark_insight "
                         "tool. Never invent numbers. After the tool returns, present its "
                         "'summary' verbatim and add one short insight."),
            tools=[insight_tool],
        )
        ADK_READY = True
        print("✅ ADK LlmAgent ready — call it via a Runner (see ADK docs).")
    else:
        print("ℹ️  GOOGLE_API_KEY not set — using the rule-based ask() agent (fully functional).")
except Exception as e:
    print(f"ℹ️  ADK layer not active ({type(e).__name__}: {e}) — rule-based ask() still works.")

---
## 8  Demo — sample Q&A (this is the transcript)

Runs several natural-language questions end-to-end: each prints the parsed tool
parameters, the **narrated summary**, and a **chart**. The final question shows
**cross-session recall** working.

In [ ]:
ask("What are the total sales by region?")

In [ ]:
ask("Show me the top 5 sub-categories by total sales")

In [ ]:
ask("What is the average profit by category?")

In [ ]:
ask("Show the monthly sales trend")

In [ ]:
ask("What are the total sales in the West region by segment?")

In [ ]:
# Cross-session recall: memory persists to disk, so re-running the notebook
# in a NEW session still shows these earlier questions.
ask("Which category has the lowest total profit?")

---
## 9  Sample Q&A transcript (for submission)

> Below is a captured transcript of the agent answering. A standalone copy is exported as
> **`sample_qa_transcript.md`** alongside this notebook.

**Q1 — "What are the total sales by region?"**
Tool params → `metric=sum, measure=sales, group_by=region`
> **Sum Of Sales by region:** West / East / Central / South listed with values; highest region highlighted. 📊 Bar chart.

**Q2 — "Show me the top 5 sub-categories by total sales"**
Tool params → `metric=sum, measure=sales, group_by=sub_category, limit=5, sort=desc`
> Top-5 sub-categories with totals; leader highlighted. 📊 Bar chart.

**Q3 — "What is the average profit by category?"**
Tool params → `metric=avg, measure=profit, group_by=category`
> Average profit per category (Furniture / Office Supplies / Technology). 📊 Bar chart.

**Q4 — "Show the monthly sales trend"**
Tool params → `metric=sum, measure=sales, group_by=month`
> Month-by-month sales. 📈 Line chart.

**Q5 — "What are the total sales in the West region by segment?"**
Tool params → `metric=sum, measure=sales, group_by=segment, filters={region: West}`
> Sales by segment within West. 📊 Bar chart.

**Q6 — "Which category has the lowest total profit?"** *(with memory recall)*
> 🧠 Memory shows Q1–Q5 from earlier in the session, then answers the new question.

*(The exact numbers depend on whether the external dataset or the synthetic fallback was
used; either way the values are computed by PySpark, never invented.)*


---
## 10  📤 Submission & GitHub backup

Per the briefing, **Colab links expire**, so back the project up on GitHub and submit the repo link.

```bash
# 1) Save a copy of this notebook, then push to a PUBLIC repo:
git init
git add PySpark_Data_Insights_Agent.ipynb sample_qa_transcript.md
git commit -m "Week 3 take-home: PySpark Data Insights Agent"
git branch -M main
git remote add origin https://github.com/<your-username>/pyspark-data-insights-agent.git
git push -u origin main
```

**Submit**
- ✅ Public **GitHub repo link** (notebook + transcript)
- ✅ `.ipynb` export of this notebook
- ✅ `sample_qa_transcript.md`
- ✅ The **external data link** you used (`DATA_URL` at the top of Section 2)

**Checklist mapped to the assignment**
| Requirement | Where |
|---|---|
| Natural-language question in | `ask()` — Section 6 |
| Translated via a **tool** to a **parameterized PySpark aggregation** | `parse_question` + `run_aggregation` — Sections 3 |
| Larger **CSV/Parquet** dataset from an **external link** | Section 2 (CSV→Parquet, external URL + fallback) |
| **Narrated summary + chart** | `narrate` + `make_chart` — Section 4 |
| *(Optional)* **Memory Bank** cross-session recall | Sections 5 & 6 |
| Colab notebook + Q&A transcript | This notebook + `sample_qa_transcript.md` |
